# Stage C — held-out evaluation and authorization gate
Evaluates separately trained controls, generates the paired-accession bootstrap assessment, and never changes model weights.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='1cd65ffe7fee0d84da315594f36c58ac6a2333d5'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
DATASET_DIR=f'{DRIVE_ROOT}/datasets/ordered_streams/LOCKED_TOKENIZER'
PILOT_NAME='c7_bounded_100mbp'
SPLIT='val'
MAX_STREAMS=0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)

In [ ]:
root=f'{DRIVE_ROOT}/runs/{PILOT_NAME}'
output=f'{root}/evaluation_{SPLIT}'
command=['seqtrainer-titans-stage-c-evaluate','--dataset-dir',DATASET_DIR,'--output-dir',output,'--split',SPLIT,'--max-streams',str(MAX_STREAMS)]
for mode in ['adaptive','reference','frozen_memory','no_memory']:
    command.extend(['--run',f'{mode}={root}/{mode}/latest.pt'])
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',f'evaluate_{SPLIT}','--repo',str(repo),'--',*command],check=True)
print('SHARE THIS DIRECTORY:',output)